# LLM-Driven Operators: `guided` and `free`

How to use misen's `guided` and `free` operators with real LLM calls.

- **`guided`**: LLM picks one block from a list of options
- **`free`**: LLM uses tools freely in a ReAct-style loop

## Setup

In [ ]:
!cd .. && uv pip install -e ".[dev]" openai --quiet

In [ ]:
import os

# Set your API key (or export OPENAI_API_KEY in terminal before starting)
# os.environ["OPENAI_API_KEY"] = "sk-..."

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY first"

## 1. Create the LLM callable

misen's `guided` and `free` take any async function matching `list[dict] → str`.
Wrap your LLM client once, use everywhere.

**Important:** Use `response_format={"type": "json_object"}` to guarantee valid JSON output.
Without it, LLMs may return multiple JSON objects or extra text, causing parse errors.

In [ ]:
from openai import AsyncOpenAI

client = AsyncOpenAI()


async def llm(messages: list[dict[str, str]]) -> str:
    """LLM callable that matches misen's LLMCallable protocol.

    Uses response_format to guarantee valid JSON output.
    """
    resp = await client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content


# Quick test
result = await llm([{"role": "user", "content": "Say hello in JSON format."}])
print(result)

## 2. `guided` — LLM selects a strategy

Given a document, the LLM picks the best analysis method from a set of options.
Each option is a misen block — the chosen one runs automatically.

In [ ]:
from misen import tool, guided, sequential


# Define analysis strategies as blocks
@tool(name="statistical_analysis", description="Analyze numerical data with statistics: mean, median, trends")
def statistical_analysis(input: dict) -> dict:
    text = input["text"]
    # Simulate statistical analysis
    return {
        "analysis_type": "statistical",
        "findings": f"Statistical analysis of: {text[:80]}...",
        "metrics": {"data_points": 150, "trend": "upward"},
    }


@tool(name="sentiment_analysis", description="Analyze emotional tone and sentiment of text")
def sentiment_analysis(input: dict) -> dict:
    text = input["text"]
    return {
        "analysis_type": "sentiment",
        "findings": f"Sentiment analysis of: {text[:80]}...",
        "sentiment": {"positive": 0.7, "negative": 0.1, "neutral": 0.2},
    }


@tool(name="keyword_extraction", description="Extract key topics, entities, and terminology from text")
def keyword_extraction(input: dict) -> dict:
    text = input["text"]
    words = text.split()
    return {
        "analysis_type": "keywords",
        "findings": f"Keyword extraction from: {text[:80]}...",
        "keywords": words[:10],
    }

In [ ]:
# guided: LLM picks the best strategy based on the document content
smart_analyzer = guided(
    llm,
    "You are a document analysis router. Based on the document content, "
    "choose the most appropriate analysis method.",
    [statistical_analysis, sentiment_analysis, keyword_extraction],
)

# Test with a numerical document
result = await smart_analyzer.run({
    "text": "Q3 revenue increased by 23% to $4.2M. Customer acquisition cost decreased "
            "from $45 to $38. Monthly active users grew from 12,000 to 18,500. "
            "Churn rate dropped to 2.1% from 3.8% in the previous quarter."
})

print(f"LLM chose: {result['__misen__']['guided_choice']}")
print(f"Analysis type: {result['analysis_type']}")
print(f"Findings: {result['findings']}")

In [ ]:
# Test with an opinion/review document — should pick sentiment
result2 = await smart_analyzer.run({
    "text": "I absolutely love this product! The interface is intuitive and beautiful. "
            "However, the loading times are frustrating and the customer support "
            "could be more responsive. Overall, I'd recommend it despite the flaws."
})

print(f"LLM chose: {result2['__misen__']['guided_choice']}")
print(f"Analysis type: {result2['analysis_type']}")

In [ ]:
# Test with a technical document — should pick keywords
result3 = await smart_analyzer.run({
    "text": "The microservices architecture uses Kubernetes for orchestration, "
            "gRPC for inter-service communication, and PostgreSQL with read replicas "
            "for data persistence. Redis handles caching and session management."
})

print(f"LLM chose: {result3['__misen__']['guided_choice']}")
print(f"Analysis type: {result3['analysis_type']}")

## 3. `guided` in a pipeline

`guided` is a block, so it composes with other operators naturally.

In [ ]:
from misen import parallel


@tool(name="summarize")
def summarize(input: dict) -> dict:
    return {"summary": f"Summary based on {input['analysis_type']} analysis: {input['findings'][:60]}..."}


# Pipeline: analyze (LLM-guided) → summarize
pipeline = sequential(smart_analyzer, summarize)

result = await pipeline.run({
    "text": "Revenue grew 15% YoY. EBITDA margin improved to 28%. Headcount: 450 FTEs."
})

print(f"Chose: {result['__misen__']['guided_choice']}")
print(f"Summary: {result['summary']}")

## 4. `free` — LLM uses tools autonomously

The LLM receives a set of tools and decides which ones to call, in what order,
and when to stop. This is a ReAct-style agent loop.

The LLM communicates via JSON:
- `{"tool": "name", "input": {...}}` to call a tool
- `{"done": true, "result": {...}}` to finish

In [ ]:
from misen import free


# Define tools the LLM can use
@tool(name="search_docs", description="Search documents by query. Input: {query: string}. Returns matching excerpts.")
def search_docs(input: dict) -> dict:
    query = input.get("query", "")
    # Simulated search results
    db = {
        "revenue": "Q3 2024 revenue: $4.2M (+23% YoY). Q2 was $3.8M. Q1 was $3.5M.",
        "users": "MAU grew from 12K to 18.5K. DAU/MAU ratio: 45%. Top market: Korea (38%).",
        "costs": "CAC dropped from $45 to $38. Server costs: $120K/mo. Total opex: $890K/mo.",
        "team": "Engineering: 28 people. Product: 8. Sales: 12. Total: 48 FTEs.",
    }
    results = [v for k, v in db.items() if query.lower() in k.lower()]
    if not results:
        results = [v for v in db.values()][:2]  # fallback: return first 2
    return {"search_results": results}


@tool(name="calculate", description="Perform a calculation. Input: {expression: string}. Returns the numeric result.")
def calculate(input: dict) -> dict:
    expr = input.get("expression", "0")
    try:
        # Safe eval for simple math
        allowed = set("0123456789.+-*/() ")
        if all(c in allowed for c in expr):
            result = eval(expr)  # noqa: S307
        else:
            result = "Invalid expression"
    except Exception as e:
        result = f"Error: {e}"
    return {"calculation_result": result}


@tool(name="format_report", description="Format findings into a structured report. Input: {title: string, findings: list[string]}.")
def format_report(input: dict) -> dict:
    title = input.get("title", "Report")
    findings = input.get("findings", [])
    report = f"# {title}\n\n"
    for i, f in enumerate(findings, 1):
        report += f"{i}. {f}\n"
    return {"report": report}

In [ ]:
# free: LLM decides which tools to call and when to stop
researcher = free(
    llm,
    "You are a business analyst. Use the available tools to research the question "
    "and produce a structured report. Search for relevant data, perform calculations "
    "if needed, then format your findings into a report.",
    [search_docs, calculate, format_report],
    max_steps=10,
)

result = await researcher.run({
    "question": "What is our revenue growth rate and how does it compare to user growth?"
})

print(f"Steps taken: {result['__misen__']['free_steps']}")
print()
if "report" in result:
    print(result["report"])
else:
    print("Keys in result:", list(result.keys()))

## 5. `free` in a pipeline

`free` is also a block — chain it with other blocks.

In [ ]:
@tool(name="classify_question")
def classify_question(input: dict) -> dict:
    """Classify the question into a category."""
    q = input["question"].lower()
    if any(w in q for w in ["revenue", "cost", "profit", "margin"]):
        return {"category": "financial"}
    elif any(w in q for w in ["user", "growth", "retention", "churn"]):
        return {"category": "growth"}
    return {"category": "general"}


# Pipeline: classify → research (free agent) 
qa_pipeline = sequential(classify_question, researcher)

result = await qa_pipeline.run({
    "question": "What are our current user metrics?"
})

print(f"Category: {result['category']}")
print(f"Agent steps: {result['__misen__']['free_steps']}")
if "report" in result:
    print(result["report"])

## 6. Mixing `guided` and `free`

Use `guided` to pick a strategy, then `free` to execute it.

In [ ]:
# Strategy blocks — each uses 'free' internally for different research approaches
financial_researcher = free(
    llm,
    "Focus on financial metrics: revenue, costs, margins. Search for financial data, "
    "calculate growth rates, and format a financial report.",
    [search_docs, calculate, format_report],
    max_steps=8,
    name="financial_researcher",
    description="Deep-dive into financial metrics and produce a financial report",
)

growth_researcher = free(
    llm,
    "Focus on growth metrics: users, engagement, retention. Search for user data "
    "and format a growth report.",
    [search_docs, calculate, format_report],
    max_steps=8,
    name="growth_researcher",
    description="Deep-dive into user growth and engagement metrics",
)

# guided picks which researcher to use
smart_qa = guided(
    llm,
    "Based on the question, choose the most appropriate research approach.",
    [financial_researcher, growth_researcher],
)

# Run
result = await smart_qa.run({
    "question": "How are our user acquisition costs trending?"
})

print(f"Guided chose: {result['__misen__']['guided_choice']}")
if "report" in result:
    print(result["report"])

## Summary

| Operator | Use case | LLM role |
|---|---|---|
| `guided(llm, prompt, [A, B, C])` | Route to the best strategy | Picks one block to run |
| `free(llm, prompt, [tools])` | Autonomous multi-step research | Calls tools, inspects results, decides when done |

Both are blocks — they compose with `sequential`, `parallel`, `branch`, and each other.

The `llm` argument is any `async (list[dict]) → str`. Swap OpenAI for Anthropic, local models, or anything else — just change the wrapper function.